# Zero-Copy LLM Inference Server (llama.cpp + 7-8B)

## Architecture
| Layer | Technology |
|-------|------------|
| Inference engine | `llama-cpp-python` (CPU build) |
| Model | Qwen2.5-7B-Instruct Q4_K_M GGUF (bartowski) |
| IPC | `shared_memory` + `memoryview` (zero-copy) |
| Serialization | Raw UTF-8 bytes (no msgpack needed for text) |
| Sync | mmap control flag (spin polling) |
| Parallelism | multiprocessing (bypasses GIL) |

## Memory requirements
| Quantization | RAM |
|---|---|
| Q4_K_M (recommended) | ~4.5 GB |
| Q5_K_M | ~5.3 GB |
| Q8_0 | ~7.7 GB |

## Runtime
- **Google Colab**: change runtime to **CPU** (Runtime > Change runtime type)
- **Local**: Python 3.10+, 8 GB+ RAM
- **EC2**: c5.2xlarge or larger recommended (8+ vCPU)

> GPU build: replace the pip install line with  
> `CMAKE_ARGS='-DGGML_CUDA=on' pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir`

---
## Step 1: Install dependencies

In [ ]:
import subprocess, sys

# Install Japanese font for matplotlib
subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'], check=False)

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'llama-cpp-python',
     '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cpu',
     'msgpack', 'huggingface_hub', 'tqdm'],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else '')
print(result.stderr[-300:] if result.returncode != 0 else '')
print('Install complete')

---
## Step 2: Download model

In [ ]:
from huggingface_hub import hf_hub_download
import os

# bartowski's repo has single-file Q4_K_M (Qwen official repo splits into shards)
MODEL_REPO = 'bartowski/Qwen2.5-7B-Instruct-GGUF'
MODEL_FILE = 'Qwen2.5-7B-Instruct-Q4_K_M.gguf'

# Alternative:
# MODEL_REPO = 'bartowski/Meta-Llama-3.1-8B-Instruct-GGUF'
# MODEL_FILE = 'Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf'

MODEL_DIR  = './models'
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_FILE)

if os.path.exists(MODEL_PATH):
    print(f'Cached: {MODEL_PATH}')
else:
    print(f'Downloading {MODEL_FILE} ...')
    MODEL_PATH = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        local_dir=MODEL_DIR,
    )
    print(f'Saved: {MODEL_PATH}')

print(f'Model size: {os.path.getsize(MODEL_PATH)/1e9:.2f} GB')

---
## Step 3: Load model & smoke test

In [ ]:
import os, time
import matplotlib
import matplotlib.font_manager as fm
from llama_cpp import Llama

# ── Japanese font setup for matplotlib ────────────────────────────
noto_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
if os.path.exists(noto_path):
    fm.fontManager.addfont(noto_path)
    matplotlib.rc('font', family='Noto Sans CJK JP')
matplotlib.rcParams['axes.unicode_minus'] = False

# ── Benchmark parameters (edit here) ─────────────────────────────
N_THREADS  = os.cpu_count() or 2
N_CTX      = 256    # keep short for speed
MAX_TOKENS = 16     # short generation = faster benchmark
N_REQUESTS = 3      # total requests per method

print(f'Threads: {N_THREADS}, ctx: {N_CTX}, max_tokens: {MAX_TOKENS}, N: {N_REQUESTS}')
print('Loading model...')
t0 = time.perf_counter()
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=N_THREADS,
    n_batch=512,
    n_gpu_layers=0,
    verbose=False,
)
load_time = time.perf_counter() - t0
print(f'Load complete ({load_time:.1f}s)')

# Smoke test
t0 = time.perf_counter()
resp = llm('Q: What is 1+1? A:', max_tokens=MAX_TOKENS, stop=['\n','Q:'], echo=False)
elapsed = time.perf_counter() - t0
text = resp['choices'][0]['text'].strip()
tok  = resp['usage']['completion_tokens']
print(f'Output : {text}')
print(f'Speed  : {tok/elapsed:.1f} tok/s  ({elapsed*1000:.0f} ms)')

---
## Step 4: Common config & helpers

In [ ]:
import socket, struct, multiprocessing, gc
import numpy as np
from multiprocessing import shared_memory, Process, Pipe

# Shared memory layout:
# [4B: prompt_len][MAX_PROMPT_BYTES: prompt UTF-8][4B: result_len][MAX_RESULT_BYTES: result UTF-8]
MAX_PROMPT_BYTES = 2048
MAX_RESULT_BYTES = 4096
SHM_TOTAL        = 4 + MAX_PROMPT_BYTES + 4 + MAX_RESULT_BYTES

TEST_PROMPTS = [
    'Q: What is the capital of Japan? A:',
    'Q: What is 2+2? A:',
    'Q: What is the chemical formula of water? A:',
]

def _unlink_shm(name):
    try:
        s = shared_memory.SharedMemory(name=name)
        s.close(); s.unlink()
    except Exception:
        pass

print(f'SHM size  : {SHM_TOTAL} bytes')
print(f'N_REQUESTS: {N_REQUESTS}')

---
## Benchmark 1 — Baseline: pickle + TCP loopback

The simplest approach. Client serializes the prompt with `pickle`, sends it over a TCP socket,
the server deserializes, runs inference, re-serializes the result, and sends it back.
**The server process loads the model from scratch each run** — this is the key cost difference vs Persistent.

Data path: user space → kernel (TCP stack) → kernel → user space = **4 copies**.

In [ ]:
import pickle

def _tcp_llm_server(model_path, n_ctx, n_threads, max_tokens, port, ready_event, n):
    from llama_cpp import Llama
    import socket, struct, pickle
    _llm = Llama(model_path=model_path, n_ctx=n_ctx, n_threads=n_threads,
                 n_batch=512, n_gpu_layers=0, verbose=False)
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(('127.0.0.1', port)); srv.listen(1)
    ready_event.set()
    conn, _ = srv.accept()
    for _ in range(n):
        raw = conn.recv(4)
        if not raw: break
        length = struct.unpack('>I', raw)[0]
        buf = b''
        while len(buf) < length:
            buf += conn.recv(length - len(buf))
        prompt = pickle.loads(buf)
        resp   = _llm(prompt, max_tokens=max_tokens, stop=['\n','Q:'], echo=False)
        text   = resp['choices'][0]['text'].strip()
        out    = pickle.dumps(text)
        conn.sendall(struct.pack('>I', len(out)) + out)
    conn.close(); srv.close()

def benchmark_tcp(n=N_REQUESTS):
    PORT  = 19200
    ready = multiprocessing.Event()
    p = Process(target=_tcp_llm_server,
                args=(MODEL_PATH, N_CTX, N_THREADS, MAX_TOKENS, PORT, ready, n))
    p.start(); ready.wait()
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect(('127.0.0.1', PORT))
    latencies, results = [], []
    for i in range(n):
        prompt = TEST_PROMPTS[i % len(TEST_PROMPTS)]
        t0     = time.perf_counter()
        payload = pickle.dumps(prompt)
        sock.sendall(struct.pack('>I', len(payload)) + payload)
        raw = sock.recv(4)
        ln  = struct.unpack('>I', raw)[0]
        buf = b''
        while len(buf) < ln:
            buf += sock.recv(ln - len(buf))
        results.append(pickle.loads(buf))
        latencies.append((time.perf_counter() - t0) * 1000)
    sock.close(); p.join()
    return latencies, results

print('Benchmark 1: pickle + TCP  (this takes several minutes on Colab CPU)')
lat_tcp, res_tcp = benchmark_tcp()
print(f'  Median : {np.median(lat_tcp):.0f} ms')
print(f'  P99    : {np.percentile(lat_tcp,99):.0f} ms')
print(f'  QPS    : {N_REQUESTS/(sum(lat_tcp)/1000):.4f}')
print(f'  Sample : {res_tcp[0]}')

---
## Benchmark 2 — Shared memory + zero-copy

Client writes the prompt directly into a shared memory buffer. No TCP stack involved.
A control byte acts as a signal flag (0=idle, 1=request, 2=response).
The server reads from the same physical RAM without any copy.

Data path: shared RAM (direct access) = **0 copies**.

> `BufferError: cannot close exported pointers exist` may appear after the cell finishes.
> This is a Python GC ordering issue and does **not** affect results — safe to ignore.

In [ ]:
SHM_IN   = 'llm_in'
SHM_OUT  = 'llm_out'
SHM_CTRL = 'llm_ctrl'

def _shm_llm_server(model_path, n_ctx, n_threads, max_tokens,
                    shm_in, shm_out, shm_ctrl, max_prompt, max_result, n):
    from llama_cpp import Llama
    from multiprocessing import shared_memory
    import struct
    _llm = Llama(model_path=model_path, n_ctx=n_ctx, n_threads=n_threads,
                 n_batch=512, n_gpu_layers=0, verbose=False)
    s_in  = shared_memory.SharedMemory(name=shm_in)
    s_out = shared_memory.SharedMemory(name=shm_out)
    s_ctl = shared_memory.SharedMemory(name=shm_ctrl)
    mv_in = memoryview(s_in.buf)
    mv_out= memoryview(s_out.buf)
    cb    = s_ctl.buf
    for _ in range(n):
        while cb[0] != 1: pass          # spin-wait for request
        p_len  = struct.unpack_from('>I', mv_in, 0)[0]
        prompt = bytes(mv_in[4:4+p_len]).decode('utf-8')
        resp   = _llm(prompt, max_tokens=max_tokens, stop=['\n','Q:'], echo=False)
        text   = resp['choices'][0]['text'].strip()
        r_b    = text.encode('utf-8')
        struct.pack_into('>I', mv_out, 0, len(r_b))
        mv_out[4:4+len(r_b)] = r_b
        cb[0] = 2                       # signal response ready
    del mv_in, mv_out
    s_in.close(); s_out.close(); s_ctl.close()

def benchmark_shm(n=N_REQUESTS):
    for name in [SHM_IN, SHM_OUT, SHM_CTRL]:
        _unlink_shm(name)
    s_in  = shared_memory.SharedMemory(create=True, size=SHM_TOTAL,  name=SHM_IN)
    s_out = shared_memory.SharedMemory(create=True, size=MAX_RESULT_BYTES+4, name=SHM_OUT)
    s_ctl = shared_memory.SharedMemory(create=True, size=1,           name=SHM_CTRL)
    s_ctl.buf[0] = 0
    mv_in = memoryview(s_in.buf)
    mv_out= memoryview(s_out.buf)
    cb    = s_ctl.buf
    p = Process(target=_shm_llm_server,
                args=(MODEL_PATH, N_CTX, N_THREADS, MAX_TOKENS,
                      SHM_IN, SHM_OUT, SHM_CTRL,
                      MAX_PROMPT_BYTES, MAX_RESULT_BYTES, n))
    p.start()
    latencies, results = [], []
    for i in range(n):
        prompt = TEST_PROMPTS[i % len(TEST_PROMPTS)]
        p_b    = prompt.encode('utf-8')
        t0     = time.perf_counter()
        struct.pack_into('>I', mv_in, 0, len(p_b))
        mv_in[4:4+len(p_b)] = p_b
        cb[0] = 1
        while cb[0] != 2: pass
        r_len  = struct.unpack_from('>I', mv_out, 0)[0]
        result = bytes(mv_out[4:4+r_len]).decode('utf-8')
        cb[0]  = 0
        latencies.append((time.perf_counter() - t0) * 1000)
        results.append(result)
    p.join()
    del mv_in, mv_out
    for s in [s_in, s_out, s_ctl]:
        try: s.close(); s.unlink()
        except Exception: pass
    gc.collect()
    return latencies, results

print('Benchmark 2: shared memory + zero-copy')
lat_shm, res_shm = benchmark_shm()
print(f'  Median : {np.median(lat_shm):.0f} ms')
print(f'  P99    : {np.percentile(lat_shm,99):.0f} ms')
print(f'  QPS    : {N_REQUESTS/(sum(lat_shm)/1000):.4f}')
print(f'  Sample : {res_shm[0]}')

---
## Benchmark 3 — Persistent server + UNIX socket

The model is loaded **once** into a long-running process.
Requests arrive over a UNIX domain socket (faster than TCP loopback — no network stack).
This eliminates the biggest single cost: model load time (~30-60s per request in Benchmark 1).

Data path: user space → UNIX kernel buffer → user space = **2 copies**, but model load = **0**.

In [ ]:
PERSIST_SOCK = '/tmp/llm_persist.sock'
SENTINEL     = b'__STOP__'

def _persistent_server(model_path, n_ctx, n_threads, max_tokens, sock_path, ready_pipe):
    from llama_cpp import Llama
    import os, socket, struct
    _llm = Llama(model_path=model_path, n_ctx=n_ctx, n_threads=n_threads,
                 n_batch=512, n_gpu_layers=0, verbose=False)
    if os.path.exists(sock_path): os.unlink(sock_path)
    srv = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    srv.bind(sock_path); srv.listen(32)
    ready_pipe.send(True)
    while True:
        conn, _ = srv.accept()
        raw = conn.recv(4)
        if not raw: conn.close(); continue
        ln = struct.unpack('>I', raw)[0]
        data = b''
        while len(data) < ln:
            data += conn.recv(ln - len(data))
        if data == SENTINEL:
            conn.close(); break
        prompt = data.decode('utf-8')
        resp   = _llm(prompt, max_tokens=max_tokens, stop=['\n','Q:'], echo=False)
        text   = resp['choices'][0]['text'].strip().encode('utf-8')
        conn.sendall(struct.pack('>I', len(text)) + text)
        conn.close()
    srv.close()
    if os.path.exists(sock_path): os.unlink(sock_path)

def _call(sock_path, prompt):
    s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    s.connect(sock_path)
    d = prompt.encode('utf-8')
    s.sendall(struct.pack('>I', len(d)) + d)
    raw = s.recv(4)
    ln  = struct.unpack('>I', raw)[0]
    buf = b''
    while len(buf) < ln:
        buf += s.recv(ln - len(buf))
    s.close()
    return buf.decode('utf-8')

def benchmark_persistent(n=N_REQUESTS):
    parent, child = Pipe()
    p = Process(target=_persistent_server,
                args=(MODEL_PATH, N_CTX, N_THREADS, MAX_TOKENS, PERSIST_SOCK, child))
    p.start(); parent.recv()
    latencies, results = [], []
    for i in range(n):
        prompt = TEST_PROMPTS[i % len(TEST_PROMPTS)]
        t0     = time.perf_counter()
        result = _call(PERSIST_SOCK, prompt)
        latencies.append((time.perf_counter() - t0) * 1000)
        results.append(result)
    # send stop signal
    s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    s.connect(PERSIST_SOCK)
    s.sendall(struct.pack('>I', len(SENTINEL)) + SENTINEL)
    s.close()
    p.join()
    return latencies, results

print('Benchmark 3: persistent server + UNIX socket')
lat_persist, res_persist = benchmark_persistent()
print(f'  Median : {np.median(lat_persist):.0f} ms')
print(f'  P99    : {np.percentile(lat_persist,99):.0f} ms')
print(f'  QPS    : {N_REQUESTS/(sum(lat_persist)/1000):.4f}')
print(f'  Sample : {res_persist[0]}')

---
## Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import os

# Re-apply font in case kernel was restarted
noto = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
if os.path.exists(noto):
    matplotlib.font_manager.fontManager.addfont(noto)
    matplotlib.rc('font', family='Noto Sans CJK JP')
matplotlib.rcParams['axes.unicode_minus'] = False

results_map = {
    'Baseline\npickle+TCP':    lat_tcp,
    'SharedMem\nZero-Copy':    lat_shm,
    'Persistent\nUnix Socket': lat_persist,
}
colors = ['#ef4444', '#22c55e', '#3b82f6']
names  = list(results_map.keys())
lats   = list(results_map.values())

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.patch.set_facecolor('#0f172a')
for ax in axes:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#334155')

# Median latency
ax = axes[0]
medians = [np.median(l) for l in lats]
bars = ax.bar(names, medians, color=colors, alpha=0.85, edgecolor='#475569')
for bar, val in zip(bars, medians):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:.0f} ms', ha='center', va='bottom', fontsize=9, color='white')
ax.set_ylabel('Median latency (ms)', color='#94a3b8')
ax.set_title('Median latency per request', color='#f1f5f9', fontweight='bold')
ax.tick_params(axis='x', labelsize=8, colors='#cbd5e1')

# QPS
ax = axes[1]
qps  = [N_REQUESTS / (sum(l)/1000) for l in lats]
bars = ax.bar(names, qps, color=colors, alpha=0.85, edgecolor='#475569')
for bar, val in zip(bars, qps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, color='white')
ax.set_ylabel('Throughput (req/s)', color='#94a3b8')
ax.set_title('Throughput (QPS)', color='#f1f5f9', fontweight='bold')
ax.tick_params(axis='x', labelsize=8, colors='#cbd5e1')

plt.suptitle('LLM Inference Server Benchmark  (Qwen2.5-7B Q4_K_M / CPU)',
             color='#f8fafc', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('llm_benchmark.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\n=== Summary ===')
base = np.median(lat_tcp)
for name, lat in results_map.items():
    m   = np.median(lat)
    p99 = np.percentile(lat, 99)
    q   = N_REQUESTS / (sum(lat)/1000)
    print(f'{name.replace(chr(10)," "):28s}  median={m:.0f}ms  p99={p99:.0f}ms  qps={q:.4f}  speedup={base/m:.2f}x')

---
## Notes: further optimization

### Why Persistent server wins on CPU
On CPU, model load takes 30-60s per run. Benchmark 1 pays this cost on **every benchmark run**.
Benchmark 3 pays it **once**. This is the dominant difference, not the transport layer.

### When does zero-copy actually matter?
| Scenario | Transport overhead share |
|---|---|
| CPU 7B, sequential | < 0.1% — inference dominates |
| GPU A100, small model | 5-20% — transport starts to matter |
| Multi-agent, N agents | Multiplied by N — significant |
| Streaming (token-by-token) | Per-token, same order as inference |

### Tuning knobs
```python
# More threads = faster on machines with more cores
N_THREADS = os.cpu_count()   # EC2 c5.2xlarge: 8 vCPU

# Larger batch = better throughput under concurrent load
llm = Llama(model_path=..., n_batch=512)

# KV cache reuse: same system prompt prefix is cached automatically
# 2nd+ requests with same prefix are significantly faster

# EC2 tip: use instance store (NVMe) for model storage, not EBS
# Load time: NVMe ~10s vs EBS ~60s for 4.5GB model
```